# 1. 이번에는 무엇을 확인하나요?

Validation에서 정한 **B2 하나만** 고정해서 Test를 평가합니다. 재학습하거나 B0/B1을 다시 비교하지 않습니다.
전체 Test가 기본 결과이고 지정 게임 제외·단독은 영향 확인용입니다.

이 Notebook의 기본 **모두 실행**은 가상 데이터 smoke만 실행합니다. 실제 Test는 열지 않습니다.
공통 정의 → smoke 설정/실행/결과 → full 설정/실행/결과 순서로 읽으세요.
Python 테스트 파일을 먼저 실행해야 하는 것은 아닙니다.


In [ ]:
import hashlib
import json
import math
import os
import platform
from collections import Counter
from pathlib import Path

import numpy as np

EXPERIMENT_ID = "human-timing-b2-test-evaluation"
ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
SOURCE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_timing_b2_test_evaluation.ipynb"
BASELINE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_timing_baseline.ipynb"
BASELINE_DEFS_SHA = "f1943f93868c31d03ff942b90a2a3effbd083f8b8801e3729848916bfba467fb"
selected_ids = {"shared-definitions", "input-definitions", "metric-definitions", "model-definitions"}
definition_cells = [{"id": c["id"], "source": "".join(c["source"])}
                    for c in json.loads(BASELINE_NOTEBOOK.read_text(encoding="utf-8"))["cells"]
                    if c["id"] in selected_ids]
definition_bytes = (json.dumps(definition_cells, ensure_ascii=False, sort_keys=True,
                              allow_nan=False, indent=2) + "\n").encode("utf-8")
if len(definition_cells) != 4 or hashlib.sha256(definition_bytes).hexdigest() != BASELINE_DEFS_SHA:
    raise ValueError("BASELINE_DEFINITIONS_CHANGED")
SHARED_NOTEBOOK_SHA = "7ccaebcb5db7c1e5c15067f3bceebc793b3ded2c81ccb414834b31d0959ad8ed"
shared_notebook = ROOT / "scripts/ml/notebooks/human_action_data_readiness.ipynb"
if hashlib.sha256(shared_notebook.read_bytes()).hexdigest() != SHARED_NOTEBOOK_SHA:
    raise ValueError("SHARED_DEPENDENCY_CHANGED")
baseline = {"__name__": "frozen_b2_shared"}
for cell in definition_cells:
    exec(compile(cell["source"], str(BASELINE_NOTEBOOK), "exec"), baseline)
baseline["shared"]["EXPERIMENT_ID"] = EXPERIMENT_ID
for name in ("require", "finite", "integer", "json_bytes", "sha256_file", "iter_rows",
             "reject_constant", "checked_paths", "publish_report", "validate_sample",
             "prepare_distribution", "crps", "distribution_summary", "forecast_values", "ACTIONS"):
    globals()[name] = baseline[name]

FROZEN_REPORT = "data/ml-experiments/human-timing-baseline/full-report-run1.json"
REPORT_SHA = "3c631c003ad1b80ae3cad653b552a3c69869dd2309ea0f27cb92968aa41ba902"
BUNDLE_SHA = "4bda9fb6e840cc997e0eae03c7a6b9e21ec809aa6a9c5a3947f3a7ffa197eddf"
DATASET = "data/datasets/human-timing-v1-20261024-full"
MANIFEST_SHA = "c203a1db7b8f2aa7014c565b371088a9614f6abb0df5632e93fe6c807712fc88"
TEST_SHA = "2c6aa13c19a40c1a1599ad6bebfbcc7ef26e3bfec39a0a2dcfeb5020811944be"
FLAGGED_GAME = "119e01c9-b982-4907-9480-2308f4604635"
EXPECTED_COUNTS = {"test_all": (33, 9859), "test_without_flagged_game": (32, 9141),
                   "flagged_game_only": (1, 718)}


def digest_json(value):
    return hashlib.sha256(json_bytes(value)).hexdigest()


def validate_bundle(bundle):
    """입력 report가 가진 원래 분포를 검증할 뿐 다시 학습하지 않는다."""
    require(isinstance(bundle, dict) and type(bundle.get("bundleSchemaVersion")) is int
            and bundle["bundleSchemaVersion"] == 1 and bundle.get("feature") == "previousAction"
            and bundle.get("label") == "recordedIntervalMs"
            and bundle.get("canonicalArtifact") is False and bundle.get("productionIntegration") is False,
            "BUNDLE_SCHEMA_INVALID")
    games = bundle.get("trainGameIds")
    require(isinstance(games, list) and games and all(isinstance(g, str) and g for g in games)
            and games == sorted(set(games)) and integer(bundle.get("trainPairs"))
            and bundle["trainPairs"] > 0 and bundle.get("fitCount") == 3, "BUNDLE_COHORT_INVALID")
    models = bundle.get("models")
    require(isinstance(models, dict) and set(models) == {"B0", "B1", "B2"}
            and isinstance(models["B2"], dict) and models["B2"], "BUNDLE_MODELS_INVALID")
    for support in (models["B0"], models["B1"], *models["B2"].values()):
        prepare_distribution(support)
        require(support == sorted(support), "BUNDLE_SUPPORT_NOT_SORTED")
    require(len(models["B0"]) == 1 and len(models["B1"]) == bundle["trainPairs"]
            and all(isinstance(a, str) and a in ACTIONS for a in models["B2"])
            and sorted(x for group in models["B2"].values() for x in group) == models["B1"],
            "BUNDLE_SUPPORT_COUNT_INVALID")
    return bundle


def cdf_points(values, weights=None):
    x = np.asarray(values, dtype=float)
    w = np.ones(len(x)) if weights is None else np.asarray(weights, dtype=float)
    support, inverse = np.unique(x, return_inverse=True)
    probability = np.cumsum(np.bincount(inverse, weights=w)) / w.sum()
    probability = np.minimum(probability, 1)
    probability[-1] = 1
    return {"ms": support.tolist(), "probability": probability.tolist()}


# 2. 같은 모델을 세 범위에서 보기

계산은 세 범위에서 같습니다. 바뀌는 것은 평가할 행의 mask뿐입니다.
CRPS는 분포 오차이며 단위가 ms이고 작을수록 좋습니다. 평균 몇 ms 틀렸다는 뜻은 아닙니다.
Primary는 전체 Test의 game-macro CRPS입니다. 제외 결과가 좋아도 전체 결과를 대체하지 않습니다.


In [ ]:
def evaluate_b2(bundle, rows):
    """B2만 score한다. 기존 fit/세 모델 비교 함수는 호출하지 않는다."""
    validate_bundle(bundle)
    require(rows, "EMPTY_EVALUATION")
    rows = sorted(rows, key=lambda r: r["sampleId"])
    games = sorted({r["gameId"] for r in rows})
    require(not set(games) & set(bundle["trainGameIds"]), "EVALUATION_GAME_LEAKAGE")
    seen = set()
    for row in rows:
        validate_sample(row, "test", set(games))
        require(row["sampleId"] not in seen, "DUPLICATE_SAMPLE")
        seen.add(row["sampleId"])
    before = digest_json(bundle)
    counts = Counter(r["previousAction"] for r in rows)
    labels = [r["recordedIntervalMs"] for r in rows]
    scores, absolute = np.empty(len(rows)), np.empty(len(rows))
    action_results = []
    for action in sorted(counts):
        indices = [i for i, r in enumerate(rows) if r["previousAction"] == action]
        support = bundle["models"]["B2"].get(action, bundle["models"]["B1"])
        observed = [labels[i] for i in indices]
        prediction = distribution_summary(support)
        scores[indices] = crps(prepare_distribution(support), observed)
        absolute[indices] = np.abs(np.asarray(observed) - prediction["medianMs"])
        action_results.append({"action": action, "pairs": len(indices), "crpsMs": float(scores[indices].mean()),
                               "predicted": prediction, "observed": distribution_summary(observed)})
    per_game = []
    for game in games:
        indices = [i for i, r in enumerate(rows) if r["gameId"] == game]
        per_game.append({"gameId": game, "pairs": len(indices), "crpsMs": float(scores[indices].mean()),
                         "medianMaeMs": float(absolute[indices].mean())})
    values, weights = forecast_values(bundle, "B2", counts)
    require(before == digest_json(bundle), "BUNDLE_MUTATED")
    return {"games": len(games), "pairs": len(rows), "gameMacroCrpsMs": math.fsum(r["crpsMs"] for r in per_game) / len(games),
            "pairMicroCrpsMs": float(scores.mean()), "medianMaeMs": float(absolute.mean()),
            "perGame": per_game, "perAction": action_results, "worstGame": max(per_game, key=lambda r: (r["crpsMs"], r["gameId"])),
            "predicted": distribution_summary(values, weights), "observed": distribution_summary(labels),
            "predictedCdf": cdf_points(values, weights), "observedCdf": cdf_points(labels),
            "unseenActionFallbackPairs": sum(n for a, n in counts.items() if a not in bundle["models"]["B2"]),
            "trainActionsNotEvaluated": sorted(set(bundle["models"]["B2"]) - set(counts)),
            "modelsScored": ["B2"], "bundleSha256": before, "evaluationFitCount": 0}


def evaluate_scopes(bundle, rows, flagged, forbidden_games=(), expected_counts=None):
    require(rows and not {r["gameId"] for r in rows} & set(forbidden_games), "EVALUATION_GAME_LEAKAGE")
    selected = [r for r in rows if r["gameId"] == flagged]
    other = [r for r in rows if r["gameId"] != flagged]
    require(selected and other, "SENSITIVITY_GROUP_EMPTY")
    scopes = {"test_all": rows, "test_without_flagged_game": other, "flagged_game_only": selected}
    if expected_counts is not None:
        require(set(expected_counts) == set(scopes) and all(
            (len({r["gameId"] for r in scopes[k]}), len(scopes[k])) == v
            for k, v in expected_counts.items()), "FULL_COHORT_MISMATCH")
    return {name: evaluate_b2(bundle, subset) for name, subset in scopes.items()}


def synthetic_inputs():
    """직접 만든 fixture다. 실제 Test나 frozen report는 읽지 않는다."""
    bundle = {"bundleSchemaVersion": 1, "feature": "previousAction", "label": "recordedIntervalMs",
              "trainGameIds": ["fixture-train"], "trainPairs": 4, "fitCount": 3,
              "models": {"B0": [100], "B1": [0, 100, 100, 200],
                         "B2": {"move_left": [0, 100], "rotate": [100, 200]}},
              "canonicalArtifact": False, "productionIntegration": False}
    rows = []
    for game, values in (("fixture-flag", (0, 0)), ("fixture-a", (100, 200)), ("fixture-b", (50, 150))):
        for i, value in enumerate(values):
            lineage = {"gameId": game, "playerNumber": 1, "pieceIndex": 1, "sourceLine": i + 1,
                       "previousInputIndex": i, "currentInputIndex": i + 1}
            rows.append({"schemaVersion": 1, **lineage, "sampleId": digest_json(lineage),
                         "previousAction": "move_left" if i == 0 else "rotate",
                         "nextActionDiagnostic": "hard_drop", "recordedIntervalMs": value, "split": "test"})
    return bundle, rows, "fixture-flag"


In [ ]:
def repository_file(relative):
    relative = Path(relative)
    require(not relative.is_absolute() and ".." not in relative.parts, "INPUT_MUST_BE_REPO_RELATIVE")
    path = (ROOT / relative).resolve(strict=True)
    require(path.is_relative_to(ROOT) and path.is_file(), "INPUT_OUTSIDE_REPOSITORY")
    return path


def load_frozen_inputs():
    """Full 전용이다. 고정 후보·manifest·Test만 읽고 Train/Validation을 fit하지 않는다."""
    report_path = repository_file(FROZEN_REPORT)
    require(sha256_file(report_path) == REPORT_SHA, "FROZEN_REPORT_SHA_MISMATCH")
    source = json.loads(report_path.read_text(encoding="utf-8"), parse_constant=reject_constant)
    require(source.get("reportSchemaVersion") == 1 and source.get("experimentId") == "human-timing-baseline"
            and source.get("runMode") == "full" and source.get("fullCohortCovered") is True
            and source.get("policyDecisionEligible") is True and source.get("testFileRead") is False
            and source.get("testEvaluated") is False, "FROZEN_REPORT_CONTRACT_INVALID")
    bundle = validate_bundle(source["candidateBundle"])
    require(digest_json(bundle) == BUNDLE_SHA == source["candidateBundleSha256"], "BUNDLE_SHA_MISMATCH")
    manifest_path = repository_file(DATASET + "/manifest.json")
    require(sha256_file(manifest_path) == MANIFEST_SHA, "MANIFEST_SHA_MISMATCH")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"), parse_constant=reject_constant)
    require(manifest.get("datasetSchemaVersion") == 1 and manifest.get("experimentId") == "human-timing-dataset"
            and manifest.get("runMode") == "full" and manifest.get("fullCohortCovered") is True
            and manifest.get("featureColumns") == ["previousAction"]
            and manifest.get("labelColumn") == "recordedIntervalMs", "MANIFEST_CONTRACT_INVALID")
    groups = manifest["split"]["gameIds"]
    require(set(groups) == {"train", "validation", "test"}
            and all(isinstance(g, list) and g and all(isinstance(x, str) and x for x in g)
                    and g == sorted(set(g)) for g in groups.values()), "GAME_LIST_INVALID")
    require(not any(set(groups[a]) & set(groups[b]) for a, b in
                    (("train", "validation"), ("train", "test"), ("validation", "test"))), "GAME_SPLIT_LEAKAGE")
    require(sorted(sum(groups.values(), [])) == manifest["cohort"]["selectedGameIds"], "COHORT_MISMATCH")
    require(bundle["trainGameIds"] == groups["train"] and bundle["trainPairs"] == manifest["files"]["train.jsonl"]["rows"],
            "FROZEN_TRAIN_COHORT_MISMATCH")
    for split in ("train", "validation"):
        require(source["cohort"][split]["gameIds"] == groups[split]
                and source["cohort"][split]["pairs"] == manifest["files"][split + ".jsonl"]["rows"],
                "FROZEN_COHORT_MISMATCH")
    require(bundle["provenance"]["manifestSha256"] == MANIFEST_SHA
            and bundle["provenance"]["trainSha256"] == manifest["files"]["train.jsonl"]["sha256"],
            "BUNDLE_PROVENANCE_MISMATCH")
    test_path = repository_file(DATASET + "/test.jsonl")
    info = manifest["files"]["test.jsonl"]
    require(sha256_file(test_path) == TEST_SHA == info["sha256"] and test_path.stat().st_size == info["bytes"],
            "TEST_SHA_OR_SIZE_MISMATCH")
    rows = []
    for _, row, kind in iter_rows(test_path):
        require(kind == "parsed", "TEST_ROW_INVALID")
        validate_sample(row, "test", set(groups["test"]))
        rows.append(row)
    require(len(rows) == info["rows"] and {r["gameId"] for r in rows} == set(groups["test"]), "TEST_COUNT_MISMATCH")
    watched = {report_path: REPORT_SHA, manifest_path: MANIFEST_SHA, test_path: TEST_SHA,
               BASELINE_NOTEBOOK: sha256_file(BASELINE_NOTEBOOK), baseline["DEPENDENCY"]: SHARED_NOTEBOOK_SHA}
    require(all(sha256_file(p) == h for p, h in watched.items()), "INPUT_CHANGED")
    return bundle, rows, set(groups["train"] + groups["validation"]), watched


def checked_output(name, mode):
    evidence = ROOT / "data/ml-experiments" / EXPERIMENT_ID
    evidence.mkdir(parents=True, exist_ok=True)
    output = Path(name)
    if not output.is_absolute():
        output = ROOT / output
    _, output = checked_paths(ROOT, str(SOURCE_NOTEBOOK.relative_to(baseline["ROOT"])), str(output), mode)
    return output


def make_report(bundle, rows, flagged, mode, forbidden_games=()):
    require(mode in ("smoke", "full"), "INVALID_RUN_MODE")
    results = evaluate_scopes(bundle, rows, flagged, forbidden_games, EXPECTED_COUNTS if mode == "full" else None)
    return {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID, "runMode": mode,
            "status": "fixed_b2_test_smoke_valid" if mode == "smoke" else "fixed_b2_test_evaluation_complete",
            "fullCohortCovered": mode == "full", "policyDecisionEligible": mode == "full",
            "canonicalResultEligible": False, "productionIntegration": False, "modelFitCount": 0,
            "historicalBundleFitCount": bundle["fitCount"], "candidateModelId": "B2",
            "candidateBundleSha256": digest_json(bundle), "scopeResults": results, "primaryScope": "test_all",
            "primaryMetric": "gameMacroCrpsMs", "testFileRead": mode == "full", "inputUnchanged": True,
            "sourceNotebookSha256": sha256_file(SOURCE_NOTEBOOK), "baselineDefinitionsSha256": BASELINE_DEFS_SHA,
            "sharedNotebookSha256": SHARED_NOTEBOOK_SHA,
            "evaluationDecision": "needs_result_review" if mode == "full" else "not_eligible",
            "testQualityPreviouslyInspected": mode == "full",
            "environment": {"python": platform.python_version(), "numpy": np.__version__},
            "limits": ["recorded_attempt_interval_not_keypress", "no_refit_or_model_reselection",
                       "no_action_hold_or_gameplay_validation", "flagged_game_origin_unverified"]}


def run_smoke(output=None):
    """가상 입력을 같은 평가 경로에 전달한다. output 미지정이면 메모리에서만 확인한다."""
    destination = checked_output(output, "smoke") if output else None
    bundle, rows, flagged = synthetic_inputs()
    report = make_report(bundle, rows, flagged, "smoke")
    if destination:
        publish_report(report, destination)
    return report


def run_full_evaluation(output):
    """사용자가 full 섹션을 켰을 때만 실행하며 어떤 fit 함수도 호출하지 않는다."""
    destination = checked_output(output, "full")
    bundle, rows, forbidden, watched = load_frozen_inputs()
    watched[SOURCE_NOTEBOOK] = sha256_file(SOURCE_NOTEBOOK)
    report = make_report(bundle, rows, FLAGGED_GAME, "full", forbidden)
    report["inputs"] = {"sourceReportSha256": REPORT_SHA, "manifestSha256": MANIFEST_SHA, "testSha256": TEST_SHA}
    require(all(sha256_file(p) == h for p, h in watched.items()), "INPUT_CHANGED")
    publish_report(report, destination)
    return report


In [ ]:
def show_results(report):
    """저장된 점수·CDF만 표시하며 모델이나 실제 Test를 다시 읽지 않는다."""
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display

    results = {k: report["scopeResults"][k] for k in ("test_all", "test_without_flagged_game", "flagged_game_only")}
    print(f"Mode: {report['runMode']} | candidate B2 | new fit count: {report['modelFitCount']}")
    if report["runMode"] == "smoke":
        print("Synthetic fixture only: real frozen model and Test were not read.")
    display(pd.json_normalize([{"scope": k, **{name: v[name] for name in
                ("games", "pairs", "gameMacroCrpsMs", "pairMicroCrpsMs", "medianMaeMs", "predicted", "observed",
                 "unseenActionFallbackPairs")}} for k, v in results.items()]).set_index("scope"))
    display(pd.DataFrame([{"scope": k, **row} for k, v in results.items() for row in v["perGame"]]))
    display(pd.json_normalize([{"scope": k, **row} for k, v in results.items() for row in v["perAction"]]))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
    for ax, (scope, values) in zip(axes, results.items()):
        endpoint = max(values["predictedCdf"]["ms"][-1], values["observedCdf"]["ms"][-1])
        for key, label in (("predictedCdf", "Frozen B2"), ("observedCdf", "Observed")):
            points = values[key]
            ax.step([0, *points["ms"], endpoint], [0, *points["probability"], 1], where="post", label=label)
        ax.set(title=f"{scope}\n{values['games']} games / {values['pairs']} pairs",
               xlabel="Recorded interval (ms)", ylabel="Cumulative probability", ylim=(0, 1.02))
        ax.legend()
    fig.suptitle(f"Frozen B2 | {report['runMode']} | forecast and observed CDF")
    plt.show()
    plt.close(fig)
    games = pd.DataFrame(results["test_all"]["perGame"]).sort_values("crpsMs")
    games["game"] = games["gameId"] if report["runMode"] == "smoke" else games["gameId"].str.slice(0, 8)
    fig, axes = plt.subplots(1, 2, figsize=(13, max(4, len(games) * .23)), layout="constrained")
    games.set_index("game")["crpsMs"].plot.barh(ax=axes[0], color="#0072B2")
    axes[0].set(title="All cohort: per-game CRPS, lower is better", xlabel="CRPS (ms)", ylabel="Game ID prefix")
    zero = pd.DataFrame([{"scope": k, "B2": v["predicted"]["zeroProbability"],
                          "Observed": v["observed"]["zeroProbability"]} for k, v in results.items()])
    zero.set_index("scope").plot.bar(ax=axes[1], rot=25)
    axes[1].set(title="Zero-ms probability", ylabel="Fraction of pairs", xlabel="")
    fig.suptitle(f"Frozen B2 | {report['runMode']} | not gameplay evidence")
    plt.show()
    plt.close(fig)


# 3. Smoke — 가상 데이터로 코드 확인

이 섹션은 실제 frozen report·dataset·Test 파일을 열지 않습니다.
학습은 하지 않고 직접 만든 작은 B2 분포와 3개 가상 게임을 평가합니다.
예를 들어 [0,100]ms 예측 분포와 관측 0ms의 CRPS는 25ms입니다.
Smoke 결과를 실제 Human 성능으로 해석하지 마세요.


In [ ]:
# 빈 값이면 report 파일을 쓰지 않아 반복 모두 실행도 기존 결과를 덮지 않습니다.
SMOKE_OUTPUT = os.environ.get("HUMAN_TIMING_B2_SMOKE_REPORT") or None


In [ ]:
SMOKE_REPORT = run_smoke(SMOKE_OUTPUT)
print(SMOKE_REPORT["status"])


In [ ]:
show_results(SMOKE_REPORT)


# 4. Full — 사용자 승인 후 실제 Test 평가

**기본은 꺼져 있습니다.** 위 smoke가 정상이고 실제 Test를 평가할 준비가 됐을 때만
아래 RUN_FULL을 True로 바꾸고 이 full 설정·실행·결과 셀을 순서대로 실행합니다.

Train 149게임의 고정 B2를 그대로 사용합니다. 전체 33게임/9,859 pair,
지정 게임 제외 32게임/9,141 pair, 단독 1게임/718 pair를 평가하며 fit은 0회입니다.
한 번 본 Test로 모델을 다시 고르지 않습니다.

출력 파일이 이미 있으면 이름을 run2 등으로 바꾸세요. 기존 결과는 삭제하거나 덮어쓰지 않습니다.


In [ ]:
RUN_FULL = False
FULL_OUTPUT = os.environ.get(
    "HUMAN_TIMING_B2_FULL_REPORT",
    str(ROOT / "data/ml-experiments/human-timing-b2-test-evaluation/full-report-run1.json"),
)


In [ ]:
FULL_REPORT = None
if RUN_FULL:
    FULL_REPORT = run_full_evaluation(FULL_OUTPUT)
    print(FULL_REPORT["status"])
else:
    print("Full skipped: RUN_FULL=False. Real Test was not opened.")


# 5. Full 결과 읽기

Primary는 test_all의 gameMacroCrpsMs이며 낮을수록 좋습니다.
제외/단독은 민감도 분석입니다. 세 결과의 차이를 보고 후보를 바꾸거나 좋은 subset만 발표하지 않습니다.
CDF는 실제 관측과 B2 예측을 함께 표시합니다. 0ms는 기록 시각 차이이지 동시 키 누름의 증거가 아닙니다.


In [ ]:
if FULL_REPORT is not None:
    show_results(FULL_REPORT)
else:
    print("Full result is not available yet. Smoke is synthetic, not Test performance.")


# 6. 한계와 다음 gate

이 결과는 기록된 처리 시도 간격의 분포 적합성입니다. 행동 선택·Hold·개인 습관·gameplay 강도·현재 40행 적용은 검증하지 않습니다.
수치 채택 threshold를 새로 만들거나 모델을 재학습하지 않고, 결과를 공유해 별도 analyze-result로 해석합니다.

Python 테스트는 개발 검사용이며 이 Notebook 실행의 선행 조건이 아닙니다.
Source output/count 검사와 사용자 실행 성공은 구분합니다. 실행 결과를 보존한 파일을 검사 통과 목적으로 지우지 않습니다.
기존 결과만 다시 보려면 정의 셀을 실행하고 JSON report를 변수에 읽은 뒤 show_results(report)를 호출합니다.
